In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#imports
import os
from tqdm import tqdm


import cv2
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# import jiwer


from torch import nn
from transformers import TrOCRProcessor, AutoTokenizer, VisionEncoderDecoderModel

from torch.quantization import quantize_dynamic
from torchvision import transforms
from torch.optim import AdamW
from torch.utils.data import DataLoader, random_split, Dataset
from sklearn.model_selection import train_test_split


In [ ]:

# Paths
base_dir = '/kaggle/input/machathon6/Machathon_Prescription_Digtalization_6.00/Machathon_Prescription_Digtalization_6.00'
train_phase_one_folder = os.path.join(base_dir, 'Train_Data_Phase1')
test_phase_one_folder = os.path.join(base_dir, 'Test_Data_Phase1')
test_phase_two_folder = os.path.join(base_dir, 'Test_Data_Phase2')

train_phase_one_sheet = os.path.join(base_dir, 'Train.xlsx')
test_phase_one_sheet = '/kaggle/input/machathon6/test_phase_1_with_entities.xlsx'
test_phase_two_sheet = '/kaggle/input/machathon6/Test_phase_2.xlsx'
train_with_entities = '/kaggle/input/machathon6/Train_With_entities.xlsx'

# Read Excel files
sheets = {
    train_phase_one_folder: pd.read_excel(train_phase_one_sheet),
    test_phase_one_folder: pd.read_excel(test_phase_one_sheet),
    test_phase_two_folder: pd.read_excel(test_phase_two_sheet),
}



In [ ]:
# Initialize the dataset list
all_data = []

# Loop through each folder and its corresponding sheet
for folder_path, df in sheets.items():
    print(f"Processing folder: {folder_path}")
    print()
    # Try to get image_id and final_text columns
    image_col = None
    text_col = None
    for col in df.columns:
        if "image" in col.lower() or "filename" in col.lower():
            image_col = col
        if "prescription" in col.lower() or "final" in col.lower():
            text_col = col

    if image_col is None or text_col is None:
        raise ValueError(f"Could not find required columns in {folder_path} sheet.")

    for _, row in tqdm(df.iterrows(), total=len(df)):
        image_id = str(row[image_col])
        text = str(row[text_col])

        # Try jpg and png
        image_path_jpg = os.path.join(folder_path, image_id + ".jpg")
        image_path_png = os.path.join(folder_path, image_id + ".png")

        if os.path.exists(image_path_jpg):
            all_data.append({"image_path": image_path_jpg, "text": text})
        elif os.path.exists(image_path_png):
            all_data.append({"image_path": image_path_png, "text": text})
        else:
            print(f"Image not found: {image_id}")

# Save combined dataset
output_df = pd.DataFrame(all_data)
output_df.to_excel("all_data_combined.xlsx", index=False)
print("Saved to all_data_combined.xlsx")

In [ ]:
output_df.shape

In [ ]:
output_df.columns


In [ ]:
#hyperparamters



In [ ]:
#train test val split 


def split_dataframe(df, train_size=0.7, val_size=0.15, test_size=0.15):
    train_df, temp_df = train_test_split(df, train_size=train_size, shuffle=True)
    
    val_size_adjusted = val_size / (val_size + test_size)  # Scale relative to temp split
    val_df, test_df = train_test_split(temp_df, train_size=val_size_adjusted, shuffle=True)
    
    return train_df, val_df, test_df

# Split the dataset
train_df, val_df, test_df = split_dataframe(output_df)
print(f'train:{train_df.shape},  test: {test_df.shape}, val: {val_df.shape}')

In [ ]:
#images preprocessing 
class HandwrittenPreprocessor:
    def __init__(self, target_size=(384, 384)):
        self.target_size = target_size

    def enhance_contrast(self, image):
        """Enhance image contrast"""
        enhancer = ImageEnhance.Contrast(image)
        return enhancer.enhance(1.5)

    def denoise(self, image):
        """Remove noise from image"""
        img_array = np.array(image)
        denoised = cv2.fastNlMeansDenoisingColored(img_array, None, 10, 10, 7, 21)
        return Image.fromarray(denoised)

    def remove_background(self, image):
        """Remove background using adaptive thresholding"""
        img_array = np.array(image)
        gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)

        binary = cv2.adaptiveThreshold(
            gray, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV, 11, 2
        )

        kernel = np.ones((2,2), np.uint8)
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

        result = cv2.cvtColor(binary, cv2.COLOR_GRAY2RGB)
        return Image.fromarray(result)


class HandwrittenExtractor:
    def __init__(self, image, padding=100):
        self.image_path = image_path
        self.padding = padding
        self.original = image
        if self.original is None:
            raise FileNotFoundError(f"Could not load image: {image_path}")
        self.processed = None
        self.mask = None
        self.cropped = None
        self.result_img = self.original.copy()
        self._process()

    def _normalize_intensity(self, image):
        return cv2.normalize(image, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)

    def _emphasize_blue(self, image):
        b, g, r = cv2.split(image)
        return cv2.subtract(b, cv2.addWeighted(r, 0.5, g, 0.5, 0))

    def _crop_handwriting(self, mask):
        y_coords, x_coords = np.where(mask == 255)
        if len(x_coords) > 0:
            x_min, x_max = np.min(x_coords), np.max(x_coords)
            y_min, y_max = np.min(y_coords), np.max(y_coords)
            print(f"[DEBUG] Handwriting bounding box -> x:({x_min}, {x_max}), y:({y_min}, {y_max})")
            x_min = max(0, x_min - self.padding)
            y_min = max(0, y_min - self.padding)
            x_max = min(self.original.shape[1], x_max + self.padding)
            y_max = min(self.original.shape[0], y_max + self.padding)
            self.cropped = self.original[y_min:y_max, x_min:x_max]
            print(f"[DEBUG] Cropped handwriting region shape: {self.cropped.shape}")
            cv2.rectangle(self.result_img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

    def _generate_mask(self, blue_emphasized):
        """
        Generate a binary mask by applying a moving average (mean blur)
        and thresholding to extract blue handwriting regions.
        """
        # Step 1: Apply moving average (mean blur)
        blurred = cv2.blur(blue_emphasized, (3, 3))  # kernel size can be tuned

        # Step 2: Normalize blurred image
        normalized_blur = cv2.normalize(blurred, None, 0, 255, cv2.NORM_MINMAX)

        # Step 3: Apply threshold to highlight regions with strong blue presence
        _, binary_mask = cv2.threshold(normalized_blur, 150, 255, cv2.THRESH_BINARY)  # threshold value can be tuned
        return binary_mask

    def _morphological_cleaning(self, binary_mask):
        """
        Clean mask using statistical filtering with sliding window:
        removes regions whose local stats deviate from global handwriting patterns.
        """
        h, w = binary_mask.shape
        window_size = 70  # can be tuned
        stride = 30  # controls overlap
        threshold_factor = 4  # how strict the filtering is

        # Global statistics
        global_mean = np.mean(binary_mask)
        global_std = np.std(binary_mask)

        # Output mask initialized to zeros
        cleaned_mask = np.zeros_like(binary_mask)

        for y in range(0, h - window_size + 1, stride):
            for x in range(0, w - window_size + 1, stride):
                window = binary_mask[y:y + window_size, x:x + window_size]
                local_mean = np.mean(window)
                local_std = np.std(window)

                # Heuristic: if local region is "ink-like", copy it
                if (
                        local_mean > global_mean * threshold_factor and
                        local_std > global_std * threshold_factor
                ):
                    cleaned_mask[y:y + window_size, x:x + window_size] = np.maximum(
                        cleaned_mask[y:y + window_size, x:x + window_size],
                        window
                    )

        return cleaned_mask

    def _process(self):
        norm = self._normalize_intensity(self.original)
        blue = self._emphasize_blue(norm)
        # blue_emphasized = cv2.GaussianBlur(blue, (5, 5), 0)
        mask = self._generate_mask(blue)
        clean = self._morphological_cleaning(mask)
        self.mask = clean
        self._crop_handwriting(self.mask)
        self.processed = {
            "normalized": norm,
            "blue_emphasized": blue,
            "binary_mask": mask,
            "clean_mask": clean,
        }

    def show_debug_plot(self):
        p = self.processed
        plt.figure(figsize=(18, 12))

        plt.subplot(2, 3, 1)
        plt.imshow(cv2.cvtColor(self.original, cv2.COLOR_BGR2RGB))
        plt.title('Original Image')
        plt.axis('off')

        plt.subplot(2, 3, 2)
        plt.imshow(cv2.cvtColor(p["normalized"], cv2.COLOR_BGR2RGB))
        plt.title('Normalized Image')
        plt.axis('off')

        plt.subplot(2, 3, 3)
        plt.imshow(p["binary_mask"], cmap='blue_image')
        plt.title(f'Mask ')
        plt.axis('off')

        plt.subplot(2, 3, 4)
        plt.imshow(p["blue_emphasized"], cmap='blue_image')
        plt.title('Blue-Emphasized')
        plt.axis('off')

        plt.subplot(2, 3, 5)
        plt.imshow(p["clean_mask"], cmap='blue_image')
        plt.title(f'Mask ')
        plt.axis('off')

        plt.subplot(2, 3, 6)
        if self.cropped is not None:
            plt.imshow(cv2.cvtColor(self.cropped, cv2.COLOR_BGR2RGB))
        else:
            plt.imshow(np.zeros((10, 10, 3)))
        plt.title('Cropped Handwriting')
        plt.axis('off')

        plt.tight_layout()
        plt.show()

    def get_cropped_image(self):
        return self.cropped

    def get_blue_cropped_image(self):
        return self._emphasize_blue(self.cropped)

    def get_clean_mask(self):
        return self.mask

    def get_visualized_result(self):
        return self.result_img


class HandwrittenBoxExtractor:
    def __init__(self, image, blue_image):
        self.original_image = image
        self.blue_image = blue_image
        self.binary = None
        self.dilated = None
        self.final_boxes = []
        
    def get_boxes(self):
        return self.final_boxes

    def preprocess(self, visualize):
        # Step 1: Binary inverse thresholding
        _, self.binary = cv2.threshold(self.blue_image, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        self.binary = cv2.bitwise_not(self.binary)
        if visualize:
            self._show_image(self.binary, "Binary Thresholded (Otsu Inverse)", cmap='gray')

        # Step 2: Morphological Opening to remove noise
        kernel_open = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
        opened = cv2.morphologyEx(self.binary, cv2.MORPH_OPEN, kernel_open, iterations=1)
        if visualize:
            self._show_image(opened, "After Morphological Opening", cmap='gray')

        # Step 3: Dilation to group characters
        kernel_dilate = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 10))
        self.dilated = cv2.dilate(opened, kernel_dilate, iterations=3)
        if visualize:
            self._show_image(self.dilated, "After Dilation", cmap='gray')

    def find_and_filter_boxes(self, visualize):
        contours, _ = cv2.findContours(self.dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        boxes = [cv2.boundingRect(c) for c in contours if cv2.contourArea(c) > 400]
        self.final_boxes = self.non_max_suppression_fast(boxes)
        if visualize:
            self.visualize_boxes(self.final_boxes, title="Refined Word Bounding Boxes", color=(0, 255, 0))

    def non_max_suppression_fast(self, boxes, overlapThresh=0.3):
        if len(boxes) == 0:
            return []

        boxes = np.array(boxes)
        pick = []

        x1 = boxes[:, 0]
        y1 = boxes[:, 1]
        x2 = boxes[:, 0] + boxes[:, 2]
        y2 = boxes[:, 1] + boxes[:, 3]

        area = boxes[:, 2] * boxes[:, 3]
        idxs = np.argsort(y2)

        while len(idxs) > 0:
            last = idxs[-1]
            pick.append(last)

            xx1 = np.maximum(x1[last], x1[idxs[:-1]])
            yy1 = np.maximum(y1[last], y1[idxs[:-1]])
            xx2 = np.minimum(x2[last], x2[idxs[:-1]])
            yy2 = np.minimum(y2[last], y2[idxs[:-1]])

            w = np.maximum(0, xx2 - xx1)
            h = np.maximum(0, yy2 - yy1)

            overlap = (w * h) / area[idxs[:-1]]
            idxs = np.delete(idxs, np.concatenate(([len(idxs) - 1], np.where(overlap > overlapThresh)[0])))

        return boxes[pick].astype("int")
    def stitch_words_in_line(self, original_image, boxes, padding=10, target_height=None, y_threshold=60):
      """
      Crop words from the image using bounding boxes, group by lines using y-value tolerance, 
      resize to uniform height, and concatenate them horizontally.
      
      Args:
          original_image (np.ndarray): The cropped handwriting image.
          boxes (list): List of (x, y, w, h) tuples for bounding boxes.
          padding (int): Horizontal padding between words.
          target_height (int): Height to resize all words to (optional).
          y_threshold (int): Tolerance to group words on the same line.
          
      Returns:
          np.ndarray: Image with all words stitched in one horizontal line (or stacked by lines if desired).
      """
      if not boxes.any():
          return np.zeros((target_height or 64, 64), dtype=np.uint8)

      # Sort boxes by y
      boxes = sorted(boxes, key=lambda b: b[1])

      # Group boxes into lines
      lines = []
      current_line = [boxes[0]]
      current_y = boxes[0][1]

      for box in boxes[1:]:
          if abs(box[1] - current_y) <= y_threshold:
              current_line.append(box)
          else:
              lines.append(current_line)
              current_line = [box]
              current_y = box[1]
      lines.append(current_line)  # Add the last line

      word_images = []

      for line_boxes in lines:
          # Sort words in the line by x (left to right)
          line_boxes = sorted(line_boxes, key=lambda b: b[0])

          for (x, y, w, h) in line_boxes:
              word_crop = original_image[y:y+h, x:x+w]
              if target_height is not None:
                  scale = target_height / word_crop.shape[0]
                  new_w = int(word_crop.shape[1] * scale)
                  word_crop = cv2.resize(word_crop, (new_w, target_height))
              word_images.append(word_crop)

      # Stitch word images horizontally with padding
      stitched_image = word_images[0]
      for word in word_images[1:]:
          spacer = np.ones((stitched_image.shape[0], padding, 3), dtype=np.uint8) * 255
          stitched_image = np.hstack((stitched_image, spacer, word))

      return stitched_image

    def visualize_boxes(self, boxes, title="Bounding Boxes", color=(0, 255, 0)):
        img_copy = self.original_image.copy()
        for (x, y, w, h) in boxes:
            cv2.rectangle(img_copy, (x, y), (x + w, y + h), color, 2)

        self._show_image(img_copy, title)

    def _show_image(self, img, title, cmap=None):
        plt.figure(figsize=(10, 6))
        if len(img.shape) == 2:
            plt.imshow(img, cmap=cmap if cmap else 'gray')
        else:
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(title)
        plt.axis("off")
        plt.show()

    def run(self, visualize=False):
        self.preprocess(visualize)
        self.find_and_filter_boxes(visualize)

In [ ]:
#dataset class

class HandwrittenOCRDataset(Dataset):
    def __init__(self, excel_file, processor, target_height=64):
        """
        Args:
            excel_file (str): Path to Excel file with two columns [image_dir, label].
            processor (transformer processor): Huggingface processor for image & text.
            target_height (int): Desired height of the image after resizing.
        """
        self.processor = processor
        self.target_height = target_height

        try:
            df = excel_file  # Assuming the excel file is passed as a path or DataFrame

            if len(df.columns) < 2:
                raise ValueError("Excel file must have at least 2 columns [image_dir, label]")

            self.examples = []
            for _, row in tqdm(df.iterrows(), total=len(df), desc="Loading dataset"):
                image_dir = str(row[0])  # Image directory
                label = str(row[1])      # Label column
                full_path = os.path.join(image_dir)  # Full image path (directory or file)

                if os.path.exists(full_path):
                    self.examples.append((full_path, label))  # Store image path and label
                else:
                    print(f"Warning: Image {full_path} not found")

            if not self.examples:
                raise Exception("No valid examples found in the Excel file")

            print(f"Successfully loaded {len(self.examples)} examples")

        except Exception as e:
            raise Exception(f"Error reading Excel file: {str(e)}")
    

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        image_filename, text = self.examples[idx]
        image_path = str(image_filename)
        print(image_path)

        try:
            bgr_image = cv2.imread(image_path)
            if bgr_image is None:
                raise ValueError("cv2.imread failed")

            # === Custom Preprocessing ===

            extractor = HandwrittenExtractor(bgr_image)
            cropped = extractor.get_cropped_image()
            blue_cropped = extractor.get_blue_cropped_image()

            box_extractor = HandwrittenBoxExtractor(cropped, blue_cropped)
            box_extractor.preprocess(visualize=False)
            box_extractor.find_and_filter_boxes(visualize=False)
            boxes = box_extractor.get_boxes()
            stitched_img = box_extractor.stitch_words_in_line(cropped, boxes, target_height=self.target_height)
            
            # === Convert to RGB PIL for processor ===
            rgb_img = cv2.cvtColor(stitched_img, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(rgb_img)

        except Exception as e:
            print(f"Error processing image {image_filename}: {str(e)}")
            if len(self.examples) > 1:
                return self.__getitem__((idx + 1) % len(self.examples))
            raise

        # === Huggingface processor ===
        pixel_values = self.processor(images=pil_img, return_tensors="pt").pixel_values
        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=128,
            truncation=True,
            return_tensors="pt"
        ).input_ids

        return {
            'pixel_values': pixel_values.squeeze(),
            'labels': labels.squeeze(),
            'text': text,
            'image':rgb_img
        }

In [ ]:
class HandwrittenTextExtractor:
    def __init__(self, vision_model_path="microsoft/trocr-base-stage1", tokenizer_model="aubmindlab/bert-base-arabertv2", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.processor = TrOCRProcessor.from_pretrained(vision_model_path)
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)
        self.processor.tokenizer = self.tokenizer

        self.model = VisionEncoderDecoderModel.from_pretrained(vision_model_path)
        self._configure_model()
        self.model.to(self.device)
        self.model.eval()

    def _configure_model(self):
        self.model.config.decoder_start_token_id = self.tokenizer.cls_token_id
        self.model.config.pad_token_id = self.tokenizer.pad_token_id
        self.model.config.eos_token_id = self.tokenizer.sep_token_id

        self.model.decoder.config.vocab_size = self.tokenizer.vocab_size
        self.model.config.vocab_size = self.tokenizer.vocab_size
        self.model.decoder.output_projection = nn.Linear(256, self.tokenizer.vocab_size)
        self.model.decoder.model.decoder.embed_tokens = nn.Embedding(self.tokenizer.vocab_size, 256, padding_idx=1)

        self.model.config.max_length = 47
        self.model.config.early_stopping = True
        self.model.config.no_repeat_ngram_size = 3
        self.model.config.length_penalty = 2.0
        self.model.config.num_beams = 8

    # need to be updateed to extract line handwritten text
    def predict_text_from_image(self, image_path):
        cropper = HandwrittenExtractor(image_path)
        cropped_img = cropper.get_cropped()
        if cropped_img is None:
            return ""

        pixel_values = self.processor(cropped_img, return_tensors="pt").pixel_values.to(self.device)

        with torch.no_grad():
            generated_ids = self.model.generate(pixel_values)
            predicted_text = self.processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return predicted_text

    def batch_predict_from_folder(self, image_folder, batch_size=4):
        results = {}
        images = []
        paths = []

        for file in sorted(os.listdir(image_folder)):
            if not file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")):
                continue

            full_path = os.path.join(image_folder, file)
            cropper = HandwrittenExtractor(full_path)
            cropped_img = cropper.get_cropped()

            if cropped_img is not None:
                images.append(cropped_img)
                paths.append(full_path)

            if len(images) == batch_size or file == sorted(os.listdir(image_folder))[-1]:
                if images:
                    pixel_values = self.processor(images, return_tensors="pt").pixel_values.to(self.device)
                    with torch.no_grad():
                        generated_ids = self.model.generate(pixel_values)
                        predicted_texts = self.processor.batch_decode(generated_ids, skip_special_tokens=True)
                    for path, text in zip(paths, predicted_texts):
                        results[path] = text
                    images, paths = [], []

        return results

In [ ]:
#evaluation
def compute_cer(predictions, labels):
    total_cer = 0
    for pred, label in zip(predictions, labels):
        pred = pred.lower()
        label = label.lower()

        distance = jiwer.compute_measures(label, pred)['substitutions'] + \
                  jiwer.compute_measures(label, pred)['deletions'] + \
                  jiwer.compute_measures(label, pred)['insertions']

        total_chars = len(label)
        if total_chars > 0:
            total_cer += distance / total_chars

    return total_cer / len(predictions)

def compute_wer(predictions, labels):
    return jiwer.wer(labels, predictions)

In [ ]:
# Instantiate your model wrapper class
text_extractor = HandwrittenTextExtractor(
    vision_model_path="microsoft/trocr-base-stage1",
    tokenizer_model="aubmindlab/bert-base-arabertv2"
)

# Access the internal model and processor
model = text_extractor.model
processor = text_extractor.processor
device = text_extractor.device


In [ ]:
train_df.shape

In [ ]:
train_dataset = HandwrittenOCRDataset(
    excel_file = train_df,
    processor = processor,
)

In [ ]:

sample_image = train_dataset[70]


        # return {
        #     'pixel_values': pixel_values.squeeze(),
        #     'labels': labels.squeeze(),
        #     'text': text
        # }
# Print sample details
print("Sample Image Tensor Shape:", sample_image['pixel_values'].shape)
print("Sample Label:", sample_image['labels'].shape)
print("Text:", sample_image['text'])
plt.imshow(sample_image['image'])  # Convert BGR to RGB for correct color display
plt.axis('off')  # Turn off axes for a cleaner display
plt.show()

In [ ]:

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_dataloader = DataLoader(eval_dataset, ,  batch_size=16)


In [ ]:
# Prepare dataset and dataloaders as before
dataset = CroppedHandwrittenDataset(
    crops_dir="/kaggle/input/your-crops-dir",
    excel_file="/kaggle/input/your-excel-file.xlsx",
    processor=processor,
    augment=True
)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_dataloader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

# Train and evaluate using existing functions
train_model(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    processor=processor,
    device=device,
    num_epochs=10,
    learning_rate=1e-5
)
